<table border=0 width="100%"><tr><td><p align="left"><img src="..\img\logo.png" align="left" width=300></p></td><td><font size=3><B>Lecture 7 实验 （郑海超）</B></font></td></tr></table>

# 实验要求
- 基于工作描述文档语料（train_corpus），应用TfidfVectorizer拟合一个vectorizer
- 使用上述拟合的vectorizer，得到工作描述文档集合的TF-IDF矩阵（transform）
- 自己写一份简历，使用上述拟合的vectorizer，得到简历的TF-IDF向量(transform)
- 应用余弦相似度，得到与你的简历匹配度最高工作职位
- 总结作业过程遇到的困难以及解决经验


**提交作业：**
- 在线提交：http://swufe.fanya.chaoxing.com/portal
- 截止日期：2025年11月12日晚上8点
- 仅仅提交**这个jupyter文档**，命名为“lecture7_experiment_学号_姓名.ipynb”，其中学号和姓名请替换为你自己的学号和姓名。


**下周我们邀请同学分享其完成作业的结果和过程经验总结**

# 基于工作描述文档语料（train corpus），应用TfidfVectorizer拟合一个vectorizer

tips
- 工作描述文档语料：job1.txt ... job8.txt
- txt文档读取，中文文本需要切词（预处理为中间有空格的文本，考虑去掉换行符）
- TfidfVectorizer，建议考虑中文定用词，停用词词典：stopwordsHIT.txt
- fit：使用工作描述作为训练语料库，拟合TfidfVectorizer

In [3]:
# 基于工作描述文档（train corpus），应用TfidfVectorizer拟合一个vectorizer变量
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import jieba

# === Step 1. 准备路径 ===
corpus = []
for i in range(1, 8):
    with open(f'job{i}.txt', 'r', encoding='utf-8') as f:
        text = f.read()
        text = text.replace('\n', '')  # 去掉换行符
        corpus.append(text)

# === Step 2. 读取停用词表 ===
with open('stopwordsHIT.txt', 'r', encoding='utf-8') as f:
    stopwords = set([w.strip() for w in f.readlines()])

# === Step 3. 中文分词 + 去停用词 ===
def preprocess(text):
    tokens = jieba.lcut(text)  # 中文分词
    tokens = [w for w in tokens if w not in stopwords and w.strip()]
    return ' '.join(tokens)

train_corpus = [preprocess(doc) for doc in corpus]

# === Step 4. 构建并拟合TF-IDF向量器 ===
vectorizer = TfidfVectorizer()
vectorizer.fit(train_corpus)

print("拟合完成！")
print("词汇表大小：", len(vectorizer.vocabulary_))




Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\Nanzheng\AppData\Local\Temp\jieba.cache
Loading model cost 0.685 seconds.
Prefix dict has been built successfully.


拟合完成！
词汇表大小： 395


# 使用上述拟合的vectorizer，得到工作描述文档集合的TF-IDF矩阵
tips
- transform
- 工作描述文档集合的TF-IDF矩阵就是工作文档的量化表示

In [4]:
# 使用上述拟合的vectorizer，得到工作描述文档集合的TF-IDF矩阵（transform）
# === Step 5. 生成TF-IDF矩阵 ===
tfidf_matrix = vectorizer.transform(train_corpus)

print("TF-IDF矩阵生成完成！")
print("矩阵形状（文档数 × 词汇数）：", tfidf_matrix.shape)

# 查看第1个文档的前10个非零特征及对应TF-IDF值
import numpy as np

doc_vector = tfidf_matrix[0]  # 第1个文档
indices = doc_vector.nonzero()[1]  # 非零列索引
values = doc_vector.data  # 对应的TF-IDF值

print("第1个文档非零特征（前10个）及对应TF-IDF：")
for idx, val in zip(indices[:10], values[:10]):
    print(vectorizer.get_feature_names_out()[idx], ":", val)

TF-IDF矩阵生成完成！
矩阵形状（文档数 × 词汇数）： (7, 395)
第1个文档非零特征（前10个）及对应TF-IDF：
一定 : 0.0921957635175755
上线 : 0.184391527035151
专业 : 0.04379459379232382
业务 : 0.06541565014053675
主管 : 0.0921957635175755
互联网 : 0.0921957635175755
产品 : 0.567943792601991
产品开发 : 0.0921957635175755
产品设计 : 0.0921957635175755
以上 : 0.04379459379232382


# 自己写一份简历，使用上述拟合的vectorizer，得到个人简历的TF-IDF矩阵

tips
- 简历用多行的字符串"""your resume"""，your resume替换为你的中文个人简历
- transform
- 个人简历的TF-IDF矩阵就是其量化表示

In [5]:
# 自己写一份简历，使用上述拟合的vectorizer，得到简历的TF-IDF向量
# === Step 6. 读取简历文本 ===
with open('resume.txt', 'r', encoding='utf-8') as f:
    resume_text = f.read()
    resume_text = resume_text.replace('\n', '')  # 去掉换行符

# === Step 7. 简历分词 + 去停用词 ===
def preprocess(text):
    tokens = jieba.lcut(text)
    tokens = [w for w in tokens if w not in stopwords and w.strip()]
    return ' '.join(tokens)

resume_processed = preprocess(resume_text)

# === Step 8. 使用拟合好的vectorizer生成TF-IDF向量 ===
resume_vector = vectorizer.transform([resume_processed])

print("简历TF-IDF向量生成完成！")
print("向量形状（1×词汇数）：", resume_vector.shape)

# 查看非零特征数量
print("非零特征数量：", resume_vector.nnz)




简历TF-IDF向量生成完成！
向量形状（1×词汇数）： (1, 395)
非零特征数量： 52


# 应用余弦相似度，得到与你的简历匹配度最高工作职位

tips
- cosine_similarity：可以使用from sklearn.metrics.pairwise import cosine_similarity，也可以自己用numpy编程实现

In [6]:
# 应用余弦相似度，得到与你的简历匹配度最高工作职位
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# === Step 9. 计算余弦相似度 ===
similarity_scores = cosine_similarity(resume_vector, tfidf_matrix)  # 1×7
similarity_scores = similarity_scores.flatten()  # 转成1维数组

# 输出每个岗位匹配度
for i, score in enumerate(similarity_scores, 1):
    print(f"job{i}.txt 匹配度: {score:.4f}")

# 找到匹配度最高的岗位
best_index = np.argmax(similarity_scores) + 1
print(f"\n与你的简历匹配度最高的职位是：job{best_index}.txt，匹配度为 {similarity_scores[best_index-1]:.4f}")



job1.txt 匹配度: 0.1576
job2.txt 匹配度: 0.3960
job3.txt 匹配度: 0.4588
job4.txt 匹配度: 0.0857
job5.txt 匹配度: 0.3093
job6.txt 匹配度: 0.2335
job7.txt 匹配度: 0.0401

与你的简历匹配度最高的职位是：job3.txt，匹配度为 0.4588


# 总结作业过程遇到的困难以及解决经验

tips
- 在下面的markdown格子中写

在这里开始-----
总结作业过程遇到的困难以及解决经验：





##### 1.写了一份简历，用了txt文本格式，所以用了读取

##### 2.没有遇到什么问题，原本用trae和deepseek遇到了很多大模型理解方向的问题，但是通过使用GPT这些问题就解决了

##### 结果挺准确的嘿嘿